In [6]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os
from pathlib import Path


PROJECT_DIR = '/content/drive/MyDrive/derma_project'
DATA_DIR = f'{PROJECT_DIR}/data/archive'
PART1 = f'{DATA_DIR}/HAM10000_images_part_1'
PART2 = f'{DATA_DIR}/HAM10000_images_part_2'
METADATA_PATH = f'{DATA_DIR}/HAM10000_metadata.csv'

print("Setup complete")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup complete


In [10]:
df = pd.read_csv(METADATA_PATH)

print(f"First rows: {df.head()}")

First rows:      lesion_id      image_id   dx dx_type   age   sex localization
0  HAM_0000118  ISIC_0027419  bkl   histo  80.0  male        scalp
1  HAM_0000118  ISIC_0025030  bkl   histo  80.0  male        scalp
2  HAM_0002730  ISIC_0026769  bkl   histo  80.0  male        scalp
3  HAM_0002730  ISIC_0025661  bkl   histo  80.0  male        scalp
4  HAM_0001466  ISIC_0031633  bkl   histo  75.0  male          ear


In [11]:
print("Diagnosis distribution:")
print(df['dx'].value_counts())
print(f"\nMissing values:")
print(df.isnull().sum())

Diagnosis distribution:
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64

Missing values:
lesion_id        0
image_id         0
dx               0
dx_type          0
age             57
sex              0
localization     0
dtype: int64


In [20]:
missing_age_df = df[df['age'].isnull()]
diagnosis_counts = missing_age_df['dx'].value_counts()
print(diagnosis_counts)

df_clean = df.dropna(subset=['age'])
df_clean.isnull().sum()

dx
nv     45
bkl    10
mel     2
Name: count, dtype: int64


,0
lesion_id,0
image_id,0
dx,0
dx_type,0
age,0
sex,0
localization,0


In [21]:
print(df_clean['sex'].value_counts())
print(f"\n")
print(df_clean['localization'].value_counts())
print(f"\n")

print(f"Age range: {df_clean['age'].min()} to {df_clean['age'].max()}")
print(f"Age dtype: {df_clean['age'].dtype}")

sex
male       5400
female     4548
unknown      10
Name: count, dtype: int64


localization
back               2192
lower extremity    2077
trunk              1403
upper extremity    1116
abdomen            1021
face                743
chest               405
foot                317
unknown             187
neck                168
scalp               128
hand                 90
ear                  56
genital              48
acral                 7
Name: count, dtype: int64


Age range: 0.0 to 85.0
Age dtype: float64


In [27]:
df_clean['age'] = df_clean['age'].astype(int)

df_clean = df_clean[df_clean['sex'] != 'unknown']

print(df_clean['sex'].value_counts())
print(f"\nAge dtype now: {df_clean['age'].dtype}")
print(f"Age range: {df_clean['age'].min()} to {df_clean['age'].max()}")

sex
male      5400
female    4548
Name: count, dtype: int64

Age dtype now: int64
Age range: 0 to 85


In [33]:
sample_sizes = {'nv': 500, 'mel': 500, 'bkl': 500, 'bcc': 300, 'akiec': 327, 'vasc': 142, 'df': 115}

bal_df = []

for diag, n in sample_sizes.items():
  samp = df_clean[df_clean['dx'] == diag]
  samp = samp.sample(n=n, random_state=42)
  bal_df.append(samp)

df_balanced = pd.concat(bal_df).reset_index(drop=True)
print(df_balanced['dx'].value_counts())
print(df_balanced.head())


dx
nv       500
mel      500
bkl      500
akiec    327
bcc      300
vasc     142
df       115
Name: count, dtype: int64
     lesion_id      image_id  dx    dx_type  age     sex localization
0  HAM_0001756  ISIC_0027484  nv  follow_up   80    male         back
1  HAM_0003624  ISIC_0025608  nv  follow_up   50  female        trunk
2  HAM_0002025  ISIC_0028023  nv      histo    5    male        scalp
3  HAM_0007004  ISIC_0030525  nv      histo   70  female         hand
4  HAM_0004149  ISIC_0033994  nv      histo   45  female         foot


In [34]:
print(f"Final dataset size: {len(df_balanced)}")
print(f"\nFinal distribution:")
print(df_balanced['dx'].value_counts())

Final dataset size: 2384

Final distribution:
dx
nv       500
mel      500
bkl      500
akiec    327
bcc      300
vasc     142
df       115
Name: count, dtype: int64


In [35]:
def find_image_path(image_id):
    filename = f"{image_id}.jpg"

    path1 = os.path.join(PART1, filename)
    path2 = os.path.join(PART2, filename)

    if os.path.exists(path1):
        return path1
    elif os.path.exists(path2):
        return path2
    else:
        return None

test_id = df_balanced['image_id'].iloc[0]
test_path = find_image_path(test_id)
print(f"Test image ID: {test_id}")
print(f"Found at: {test_path}")

Test image ID: ISIC_0027484
Found at: /content/drive/MyDrive/derma_project/data/archive/HAM10000_images_part_1/ISIC_0027484.jpg


In [37]:
df_balanced['image_path'] = df_balanced['image_id'].apply(find_image_path)

missing = df_balanced['image_path'].isnull().sum()
print(f"Images found: {len(df_balanced) - missing}")
print(f"Images missing: {missing}")

Images found: 2384
Images missing: 0


In [39]:
!pip install google-genai -q

In [40]:
from google import genai
from google.genai import types
from google.colab import userdata
from PIL import Image
import time
import json

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

print("Gemini client ready")

Gemini client ready


In [43]:
import re

def clean_report(text):
    text = re.sub(r'\*\*(.+?)\*\*', r'\1', text)
    text = re.sub(r'\*(.+?)\*', r'\1', text)
    return text.strip()



def generate_dermatology_report(image_path, age, sex, localization, diagnosis):

    diagnosis_names = {
        'mel':   'melanoma',
        'nv':    'melanocytic nevus',
        'bcc':   'basal cell carcinoma',
        'akiec': 'actinic keratosis',
        'bkl':   'benign keratosis',
        'vasc':  'vascular lesion',
        'df':    'dermatofibroma'
    }

    full_diagnosis = diagnosis_names[diagnosis]

    image = Image.open(image_path).convert("RGB")

    prompt = f"""You are an expert dermatologist writing a clinical analysis report.

Patient information:
- Age: {age} years old
- Sex: {sex}
- Lesion location: {localization}
- Confirmed diagnosis: {full_diagnosis}

Analyze the dermatoscopic image and write a structured clinical report.
The diagnosis has already been confirmed — your job is to describe the
visual features in the image that support this diagnosis.

Your report must follow this exact format:

DERMATOSCOPIC ANALYSIS REPORT

Patient: {age}-year-old {sex}, lesion located on {localization}.

Visual features observed:
- [describe 3-4 specific visual features visible in the image]

Assessment: The visual features are consistent with {full_diagnosis}.
[1-2 sentences explaining how the features support the diagnosis]

Recommendation: [appropriate clinical recommendation for this diagnosis]"""

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=[prompt, image]
    )

    return clean_report(response.text)

print("Function defined")

Function defined


In [44]:
row = df_balanced.iloc[0]

report = generate_dermatology_report(
    image_path=row['image_path'],
    age=row['age'],
    sex=row['sex'],
    localization=row['localization'],
    diagnosis=row['dx']
)

print(f"Diagnosis: {row['dx']}")
print(f"Image: {row['image_id']}")
print(f"\nGenerated report:")
print(report)

Diagnosis: nv
Image: ISIC_0027484

Generated report:
DERMATOSCOPIC ANALYSIS REPORT

Patient: 80-year-old male, lesion located on back.

Visual features observed:
- Predominant brown pigment network: The lesion largely consists of a brown pigment network, varying from a fine and delicate pattern at the periphery to a thicker, denser, and somewhat irregular appearance centrally, occasionally forming confluent dark brown areas.
- Multiple shades of brown and black pigment: A spectrum of colors is present, ranging from light tan at the edges to dark brown and blackish hues within the denser areas, indicative of varying melanin depth and concentration.
- Focal areas of regression/hypopigmentation: Whitish-pinkish areas are observed, particularly prominent in the inferior and central aspects, suggesting areas of benign regression or fibrosis.
- Presence of a small, elevated papular component: A distinct, slightly raised area with a faint, fine pigment network is noted at the superior aspect 

In [47]:
import json
import time

OUTPUT_PATH = f'{PROJECT_DIR}/data/generated_reports.json'

if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, 'r') as f:
        reports = json.load(f)
    completed_ids = {r['image_id'] for r in reports}
    print(f"Resuming — {len(reports)} reports already generated")
else:
    reports = []
    completed_ids = set()
    print("Starting fresh")

df_todo = df_balanced[~df_balanced['image_id'].isin(completed_ids)].copy()
print(f"Remaining: {len(df_todo)} reports to generate")

Starting fresh
Remaining: 2384 reports to generate


In [50]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

lock = threading.Lock()

def process_row(row_data):
    _, row = row_data
    try:
        report_text = generate_dermatology_report(
            image_path=row['image_path'],
            age=row['age'],
            sex=row['sex'],
            localization=row['localization'],
            diagnosis=row['dx']
        )
        return {
            'image_id':     row['image_id'],
            'image_path':   row['image_path'],
            'age':          row['age'],
            'sex':          row['sex'],
            'localization': row['localization'],
            'diagnosis':    row['dx'],
            'report':       report_text
        }
    except Exception as e:
        print(f"Error on {row['image_id']}: {e}")
        return None

WORKERS = 10
completed_count = len(reports)

with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = {
        executor.submit(process_row, row_data): row_data
        for row_data in df_todo.iterrows()
    }

    for future in as_completed(futures):
        result = future.result()
        if result:
            with lock:
                reports.append(result)
                completed_count += 1

                if completed_count % SAVE_EVERY == 0:
                    with open(OUTPUT_PATH, 'w') as f:
                        json.dump(reports, f)
                    print(f"Checkpoint — {completed_count} reports complete")

with open(OUTPUT_PATH, 'w') as f:
    json.dump(reports, f)

print(f"Done. Total: {len(reports)}")

Checkpoint — 250 reports complete
Error on ISIC_0030634: expected string or bytes-like object, got 'NoneType'
Checkpoint — 300 reports complete
Checkpoint — 350 reports complete
Checkpoint — 400 reports complete
Checkpoint — 450 reports complete
Checkpoint — 500 reports complete
Checkpoint — 550 reports complete
Error on ISIC_0027182: expected string or bytes-like object, got 'NoneType'
Checkpoint — 600 reports complete
Checkpoint — 650 reports complete
Checkpoint — 700 reports complete
Checkpoint — 750 reports complete
Checkpoint — 800 reports complete
Error on ISIC_0034313: expected string or bytes-like object, got 'NoneType'
Checkpoint — 850 reports complete
Checkpoint — 900 reports complete
Error on ISIC_0026158: expected string or bytes-like object, got 'NoneType'
Checkpoint — 950 reports complete
Checkpoint — 1000 reports complete
Checkpoint — 1050 reports complete
Error on ISIC_0032182: expected string or bytes-like object, got 'NoneType'
Checkpoint — 1100 reports complete
Check

In [56]:
print(f"Total reports in list: {len(reports)}")

image_ids = [r['image_id'] for r in reports]
unique_ids = set(image_ids)

print(f"Unique image IDs: {len(unique_ids)}")
print(f"Duplicates: {len(reports) - len(unique_ids)}")

Total reports in list: 2582
Unique image IDs: 2376
Duplicates: 206


In [57]:
seen = set()
reports_clean = []

for r in reports:
    if r['image_id'] not in seen:
        seen.add(r['image_id'])
        reports_clean.append(r)

print(f"Clean dataset size: {len(reports_clean)}")
print(f"Removed duplicates: {len(reports) - len(reports_clean)}")

CLEAN_OUTPUT_PATH = f'{PROJECT_DIR}/data/generated_reports_clean.json'
with open(CLEAN_OUTPUT_PATH, 'w') as f:
    json.dump(reports_clean, f)

print(f"Clean dataset saved to Drive")

Clean dataset size: 2376
Removed duplicates: 206
Clean dataset saved to Drive


In [58]:
generated_ids = {r['image_id'] for r in reports_clean}
balanced_ids = set(df_balanced['image_id'])
failed_ids = balanced_ids - generated_ids

print(f"Total expected: {len(df_balanced)}")
print(f"Total generated: {len(reports_clean)}")
print(f"Missing: {len(failed_ids)}")

Total expected: 2384
Total generated: 2376
Missing: 8


In [59]:
for image_id in failed_ids:
    row = df_balanced[df_balanced['image_id'] == image_id].iloc[0]
    path = row['image_path']
    print(f"{image_id} | {row['dx']} | exists: {os.path.exists(path)}")

ISIC_0027182 | nv | exists: True
ISIC_0034313 | mel | exists: True
ISIC_0030634 | nv | exists: True
ISIC_0026158 | mel | exists: True
ISIC_0032182 | mel | exists: True
ISIC_0032775 | vasc | exists: True
ISIC_0033031 | vasc | exists: True
ISIC_0033536 | akiec | exists: True


In [60]:
generated_ids = {r['image_id'] for r in reports_clean}
df_final = df_balanced[df_balanced['image_id'].isin(generated_ids)].copy()

print(f"Final training set: {len(df_final)} examples")
print(f"\nFinal distribution:")
print(df_final['dx'].value_counts())

df_final.to_csv(f'{PROJECT_DIR}/data/df_final.csv', index=False)
print("\nFinal dataframe saved")

Final training set: 2376 examples

Final distribution:
dx
bkl      500
nv       498
mel      497
akiec    326
bcc      300
vasc     140
df       115
Name: count, dtype: int64

Final dataframe saved


In [61]:
import random

samples = random.sample(reports_clean, 3)

for s in samples:
    print(f"ID: {s['image_id']}")
    print(f"Diagnosis: {s['diagnosis']}")
    print(f"Patient: {s['age']}-year-old {s['sex']}, {s['localization']}")
    print(f"Report preview: {s['report'][:150]}")
    print("---")

ID: ISIC_0032349
Diagnosis: akiec
Patient: 60-year-old male, lower extremity
Report preview: DERMATOSCOPIC ANALYSIS REPORT

Patient: 60-year-old male, lesion located on lower extremity.

Visual features observed:
-   Rosette-like vascular patt
---
ID: ISIC_0024635
Diagnosis: bkl
Patient: 60-year-old female, face
Report preview: DERMATOSCOPIC ANALYSIS REPORT

Patient: 60-year-old female, lesion located on face.

Visual features observed:
- Multiple dark brown to blackish, roun
---
ID: ISIC_0029412
Diagnosis: bcc
Patient: 80-year-old male, back
Report preview: DERMATOSCOPIC ANALYSIS REPORT

Patient: 80-year-old male, lesion located on back.

Visual features observed:
-   Arborizing telangiectasias: Fine, bra
---
